# Graph Inspection Notebook

This notebook loads and inspects the knowledge graph using the utility functions from `cli/commands/evals/utils.py`.

In [1]:
from pathlib import Path
import sys

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

from cli.commands.evals.utils import (
    load_graph,
    load_node_metadata,
    compute_pagerank,
    pagerank_to_dataframe,
)

[03/10/26 01:33:35] INFO     Using 'conf/logging.yml' as logging configuration. You can change this ]8;id=529621;file:///Users/an583/Documents/Zitnik_Lab/optimuskg/.venv/lib/python3.12/site-packages/kedro/framework/project/__init__.py\__init__.py]8;;\:]8;id=115424;file:///Users/an583/Documents/Zitnik_Lab/optimuskg/.venv/lib/python3.12/site-packages/kedro/framework/project/__init__.py#270\270]8;;\
                             by setting the KEDRO_LOGGING_CONFIG environment variable accordingly.                 

## Load Graph

In [5]:
nodes_path = Path("data/gold/kg/parquet/nodes.parquet")
edges_path = Path("data/gold/kg/parquet/edges.parquet")

G, edge_type_lookup, node_types, edge_types = load_graph(nodes_path, edges_path)

[03/10/26 01:34:07] INFO     Loaded 192123 nodes with 10 node types: ['ANA', 'BPO', 'CCO', 'DIS',       ]8;id=261949;file:///Users/an583/Documents/Zitnik_Lab/optimuskg/cli/commands/evals/utils.py\utils.py]8;;\:]8;id=953868;file:///Users/an583/Documents/Zitnik_Lab/optimuskg/cli/commands/evals/utils.py#42\42]8;;\
                             'DRG', 'EXP', 'GEN', 'MFN', 'PHE', 'PWY']                                             

[03/10/26 01:35:30] INFO     Loaded 21851915 edges with 26 edge types: ['ANA-ANA', 'ANA-PRO',           ]8;id=10620;file:///Users/an583/Documents/Zitnik_Lab/optimuskg/cli/commands/evals/utils.py\utils.py]8;;\:]8;id=99156;file:///Users/an583/Documents/Zitnik_Lab/optimuskg/cli/commands/evals/utils.py#61\61]8;;\
                             'BPO-BPO', 'BPO-PRO', 'CCO-CCO', 'CCO-PRO', 'DIS-DIS', 'DIS-PHE',                     
                             'DIS-PRO', 'DRG-DIS', 'DRG-DRG', 'DRG-PHE', 'DRG-PRO', 'EXP-BPO',                     
                             'EXP-CCO', 'EXP-DIS', 'EXP-EXP', 'EXP-MFN', 'EXP-PRO', 'MFN-MFN',                     
                             'MFN-PRO', 'PHE-PHE', 'PHE-PRO', 'PRO-PRO', 'PWY-PRO', 'PWY-PWY']                     

## Basic Statistics

In [6]:
print(f"Nodes: {G.number_of_nodes():,}")
print(f"Edges: {G.number_of_edges():,}")
print(f"\nNode types ({len(node_types)}): {node_types}")
print(f"\nEdge types ({len(edge_types)}): {edge_types}")

Nodes: 192,123
Edges: 21,851,915

Node types (10): ['ANA', 'BPO', 'CCO', 'DIS', 'DRG', 'EXP', 'GEN', 'MFN', 'PHE', 'PWY']

Edge types (26): ['ANA-ANA', 'ANA-PRO', 'BPO-BPO', 'BPO-PRO', 'CCO-CCO', 'CCO-PRO', 'DIS-DIS', 'DIS-PHE', 'DIS-PRO', 'DRG-DIS', 'DRG-DRG', 'DRG-PHE', 'DRG-PRO', 'EXP-BPO', 'EXP-CCO', 'EXP-DIS', 'EXP-EXP', 'EXP-MFN', 'EXP-PRO', 'MFN-MFN', 'MFN-PRO', 'PHE-PHE', 'PHE-PRO', 'PRO-PRO', 'PWY-PRO', 'PWY-PWY']


## Load Node Metadata

In [7]:
node_metadata = load_node_metadata(nodes_path)
node_metadata.head(10)

[03/10/26 06:31:07] INFO     Loaded metadata for 192682 nodes                                           ]8;id=131423;file:///Users/an583/Documents/Zitnik_Lab/optimuskg/cli/commands/evals/utils.py\utils.py]8;;\:]8;id=176648;file:///Users/an583/Documents/Zitnik_Lab/optimuskg/cli/commands/evals/utils.py#98\98]8;;\

id,label,name
str,str,str
"""ENSG00000000003""","""GEN""","""tetraspanin 6"""
"""ENSG00000000005""","""GEN""","""tenomodulin"""
"""ENSG00000000419""","""GEN""","""dolichyl-phosphate mannosyltra…"
"""ENSG00000000457""","""GEN""","""SCY1 like pseudokinase 3"""
"""ENSG00000000460""","""GEN""","""FIGNL1 interacting regulator o…"
"""ENSG00000000938""","""GEN""","""FGR proto-oncogene, Src family…"
"""ENSG00000000971""","""GEN""","""complement factor H"""
"""ENSG00000001036""","""GEN""","""alpha-L-fucosidase 2"""
"""ENSG00000001084""","""GEN""","""glutamate-cysteine ligase cata…"


In [8]:
# Node counts by type
node_metadata.group_by("label").len().sort("len", descending=True)

label,len
str,u32
"""GEN""",61306
"""DIS""",36992
"""BPO""",25754
"""PHE""",19341
"""DRG""",16766
"""ANA""",14624
"""MFN""",10161
"""CCO""",4052
"""PWY""",2805


## Sample Edges with Types

In [ ]:
import random

# Sample 10 random edges
sample_edges = random.sample(list(G.edges()), 10)

for u, v in sample_edges:
    edge_label = edge_type_lookup.get((u, v), "N/A")
    print(f"{u} --[{edge_label}]--> {v}")

## Degree Distribution

In [ ]:
import polars as pl

degrees = [G.degree(n) for n in G.nodes()]
degree_df = pl.DataFrame({"degree": degrees})
degree_df.describe()

In [ ]:
# Top 10 nodes by degree
node_degrees = [(n, G.degree(n)) for n in G.nodes()]
top_degree = sorted(node_degrees, key=lambda x: x[1], descending=True)[:10]

for node_id, degree in top_degree:
    meta = node_metadata.filter(pl.col("id") == node_id)
    name = meta["name"][0] if meta.height > 0 else "N/A"
    label = meta["label"][0] if meta.height > 0 else "N/A"
    print(f"{node_id} ({label}): {name} - degree {degree:,}")

## Compute PageRank (Optional - takes ~30s)

In [ ]:
# Uncomment to compute PageRank
# scores = compute_pagerank(G, alpha=0.85)
# pagerank_df = pagerank_to_dataframe(scores, node_metadata)
# pagerank_df.head(20)

## Explore Specific Node

In [ ]:
def inspect_node(node_id: str):
    """Inspect a specific node and its neighbors."""
    # Get metadata
    meta = node_metadata.filter(pl.col("id") == node_id)
    if meta.height == 0:
        print(f"Node {node_id} not found in metadata")
        return
    
    print(f"Node: {node_id}")
    print(f"Type: {meta['label'][0]}")
    print(f"Name: {meta['name'][0]}")
    print(f"Degree: {G.degree(node_id):,}")
    
    # Get neighbors
    neighbors = list(G.neighbors(node_id))[:20]  # Limit to 20
    print(f"\nFirst {len(neighbors)} neighbors:")
    for neighbor in neighbors:
        edge_label = edge_type_lookup.get((node_id, neighbor), "N/A")
        n_meta = node_metadata.filter(pl.col("id") == neighbor)
        n_name = n_meta["name"][0] if n_meta.height > 0 else "N/A"
        n_type = n_meta["label"][0] if n_meta.height > 0 else "N/A"
        print(f"  --[{edge_label}]--> {neighbor} ({n_type}): {n_name}")

In [ ]:
# Example: inspect a random node
sample_node = random.choice(list(G.nodes()))
inspect_node(sample_node)